# 04 - Model Evaluation and Results

Đánh giá chi tiết hiệu suất mô hình và tạo báo cáo kết quả

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

sys.path.insert(0, os.path.abspath('../src'))

from data.load_data import load_processed_data
from models.train_model import ModelTrainer
from models.evaluate import ModelEvaluator
from models.predict_model import ModelPredictor

## 2. Load Data and Train Best Model

In [ ]:
# Load processed data
processed_data_path = '../data/processed/house_prices_processed.csv'
df = load_processed_data(processed_data_path)

# Prepare data
if 'price' in df.columns:
    target_col = 'price'
else:
    target_col = df.columns[-1]

X = df.drop(columns=[target_col])
y = df[target_col]

# Initialize trainer
trainer = ModelTrainer(random_state=42)
trainer.prepare_data(X, y, test_size=0.2)

## 3. Train Best Model (Random Forest)

In [ ]:
# Train the best model (based on previous results)
trainer.train_random_forest(n_estimators=100, max_depth=10)
y_pred = trainer.model.predict(trainer.X_test)

# Evaluate on training set
y_train_pred = trainer.model.predict(trainer.X_train)
metrics_train = ModelEvaluator.evaluate_model(trainer.y_train, y_train_pred)

# Evaluate on test set
metrics_test = ModelEvaluator.evaluate_model(trainer.y_test, y_pred)

print("\nTraining Metrics:")
for metric, value in metrics_train.items():
    print(f"{metric}: {value:.4f}")

print("\nTest Metrics:")
for metric, value in metrics_test.items():
    print(f"{metric}: {value:.4f}")

## 4. Visualize Predictions vs Actual

In [ ]:
# Create predictions vs actual plot
plt.figure(figsize=(10, 6))
plt.scatter(trainer.y_test, y_pred, alpha=0.6)
plt.plot([trainer.y_test.min(), trainer.y_test.max()], 
         [trainer.y_test.min(), trainer.y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title('Predictions vs Actual Values')
plt.tight_layout()
plt.savefig('../reports/figures/predictions_vs_actual.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Analyze Residuals

In [ ]:
# Calculate residuals
residuals = trainer.y_test - y_pred

# Create residuals plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals scatter plot
axes[0].scatter(y_pred, residuals, alpha=0.6)
axes[0].axhline(y=0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted Values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals Plot')

# Residuals histogram
axes[1].hist(residuals, bins=30, edgecolor='black')
axes[1].set_xlabel('Residuals')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Residuals')

plt.tight_layout()
plt.savefig('../reports/figures/residuals_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nResiduals Statistics:")
print(f"Mean: {residuals.mean():.4f}")
print(f"Std: {residuals.std():.4f}")
print(f"Min: {residuals.min():.4f}")
print(f"Max: {residuals.max():.4f}")

## 6. Feature Importance

In [ ]:
# Get feature importance
feature_importance = pd.DataFrame({
    'feature': trainer.feature_names,
    'importance': trainer.model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10))

# Plot feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'].head(10), feature_importance['importance'].head(10))
plt.xlabel('Importance')
plt.title('Top 10 Most Important Features')
plt.tight_layout()
plt.savefig('../reports/figures/feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Save Model

In [ ]:
# Save the trained model
model_path = '../models/house_price_model.pkl'
trainer.save_model(model_path)
print(f"Model saved to {model_path}")

## 8. Generate Report

In [ ]:
# Generate results report
report = f"""# House Price Prediction Model - Results Report

## Model: Random Forest Regressor

### Training Metrics
- RMSE: {metrics_train['RMSE']:.4f}
- MAE: {metrics_train['MAE']:.4f}
- R²: {metrics_train['R2']:.4f}
- MAPE: {metrics_train['MAPE']:.4f}

### Test Metrics
- RMSE: {metrics_test['RMSE']:.4f}
- MAE: {metrics_test['MAE']:.4f}
- R²: {metrics_test['R2']:.4f}
- MAPE: {metrics_test['MAPE']:.4f}

### Model Configuration
- Number of Estimators: 100
- Max Depth: 10
- Random State: 42

### Data Information
- Training Samples: {len(trainer.X_train)}
- Test Samples: {len(trainer.X_test)}
- Number of Features: {len(trainer.feature_names)}

### Top Features
{feature_importance.head(5).to_string()}

## Conclusion
The Random Forest model shows good performance with R² of {metrics_test['R2']:.4f} on the test set.
The model captures approximately {metrics_test['R2']*100:.2f}% of the variance in house prices.
"""

# Save report
with open('../reports/results.md', 'w', encoding='utf-8') as f:
    f.write(report)

print("Report saved to reports/results.md")
print("\nReport Preview:")
print(report)